#  FEATURE SELECTION

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

## IMPORT PACKAGES

In [41]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance

# Autocomplete settings
%config IPCompleter.greedy=True

## IMPORT DATA

In [86]:
project_path = Path ('/Users/gloriadelriomarquez/Documents/Career/Projects/ml-lab/client_churn')
df_file_name = 'modeling_dataset.pickle'

df = pd.read_pickle(project_path / "02_data" / "train" / df_file_name)

## SUPERVISED METHODS

In [87]:
# Prepare X and y
target = 'is_churn'
X = df.drop(columns=target).copy()
y = df[target].copy()

In [88]:

import pandas as pd
import plotly.express as px

# Comprobar datos
print("Dimensiones:", super_ranking.shape)
print("Tipos de datos:")
print(super_ranking[["variable", "puntos"]].dtypes)

# Preparar datos
df_plot = (
    super_ranking
    .sort_values("puntos", ascending=False)
    .reset_index(drop=True)
    .copy()
)

df_plot["variable"] = df_plot["variable"].astype(str)
df_plot["puntos"] = pd.to_numeric(
    df_plot["puntos"], errors="coerce"
).astype("float64")

df_plot = df_plot.dropna(subset=["puntos"])
df_plot["rank"] = df_plot.index

print("\nPrimeras filas:")
print(df_plot.head())

# Crear gráfico
fig = px.bar(
    df_plot,
    y="variable",
    x="puntos",
    orientation="h",
    title="<b>Ranking de variables</b>",
    height=800,
    width=700,
    hover_data=["rank", "variable", "puntos"]
)

fig.update_layout(
    template="simple_white",
    yaxis_title="",
    xaxis_title="Puntos"
)

# Mostrar dentro de JupyterLab
fig.show(renderer="iframe")


Dimensiones: (79, 8)
Tipos de datos:
variable      str
puntos      int64
dtype: object

Primeras filas:
                          variable  importancia_mi  ranking_mi  ranking_rfe  \
0  has_multi_non_cancelled_day_2.0        0.001818          57           40   
1                        city_19.0        0.000191          70           33   
2      last_day_all_auto_renew_2.0        0.001671          59           37   
3    has_multi_transaction_day_2.0        0.001902          56           38   
4                        city_16.0        0.000000          76           25   

   importancia_per  ranking_per  puntos  ranking_tot  rank  
0         0.000000           76   173.0           78     0  
1         0.000006           68   171.0           77     1  
2         0.000000           74   170.0           76     2  
3         0.000000           75   169.0           75     3  
4         0.000010           67   168.0           74     4  


### Mutual Information

In [46]:
#### Function to display the result
import plotly.express as px

def ranking_mutual_information(mutual_selector, X, num_vars=15, modo='tabla'):
    # Build ranking
    ranking_df = pd.DataFrame({
        'variable': X.columns,
        'importancia_mi': mutual_selector
    }).sort_values(by='importancia_mi', ascending=False)
    ranking_df['ranking_mi'] = np.arange(0, ranking_df.shape[0])
    
    # Table → normal output
    if modo == 'tabla':
        return ranking_df
        
    # Plot → Plotly Express
    else:
        top = ranking_df.head(num_vars).sort_values(by='importancia_mi')  # order for barh

        fig = px.bar(
            top,
            x='importancia_mi',
            y='variable',
            orientation='h',
            title=f'Top {num_vars} Variables by Mutual Information',
            labels={'importancia_mi': 'Mutual Information', 'variable': 'Variable'}
        )

        fig.update_layout(height=40 * num_vars, width=1200)
        fig.show(renderer='iframe')

        return fig

In [47]:
# Compute and review
mutual_selector = mutual_info_classif(X, y)

# Table format
rank_mi = ranking_mutual_information(mutual_selector, X)

# Plot format
ranking_mutual_information(mutual_selector, X, num_vars=20, modo='grafico');

In [48]:
variable_limit_position = 30
selected_mi = rank_mi.iloc[0:variable_limit_position].variable

# Extract the names of the selected variables
selected_mi.to_list()

# Create the dataframe with the selected features
X_mi = X[selected_mi].copy()

### Recursive Feature Elimination

In [49]:
# Instantiate
rfe = RFE(estimator=XGBClassifier(n_jobs=-1, eval_metric='auc'))

# Train
rfe.fit(X, y)

# Extract the names of the selected variables
rank_rfe = pd.DataFrame({
    'variable': X.columns,
    'ranking_rfe': rfe.ranking_
}).sort_values(by='ranking_rfe')

In [50]:
rank_rfe.head(20).T

,0,35,38,77,41,42,44,47,48,50,51,52,53,54,56,57,58,60,61,62
variable,age_unknown,has_multi_non_cancelled_day_0.0,has_multi_transaction_day_0.0,total_non_cancelled_amount_rs,last_day_all_auto_renew_0.0,last_day_all_auto_renew_1.0,last_day_any_auto_renew_0.0,has_mixed_cancel_day_0.0,has_mixed_cancel_day_1.0,age_rs,auto_renew_rate_rs,avg_amount_paid_rs,avg_payment_per_30d_rs,avg_plan_days_rs,cancel_rate_rs,days_to_expiry_rs,days_to_max_expiry_rs,last_day_max_payment_rs,last_day_n_transactions_rs,max_plan_days_rs
ranking_rfe,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [51]:
selected_rfe = X.columns[rfe.support_]

# Create dataframe with the selected features
X_rfe = X[selected_rfe].copy()

### Permutation Importance

In [52]:
# Function to display the result
def ranking_per(predictors, permutation):
    ranking_per = pd.DataFrame({
        'variable': predictors.columns,
        'importancia_per': permutation.importances_mean
    }).sort_values(by='importancia_per', ascending=False)
    
    ranking_per['ranking_per'] = np.arange(0, ranking_per.shape[0])
    return ranking_per

In [53]:
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier 

# Instantiate and train
xgb = XGBClassifier(n_jobs=-1, eval_metric='auc')

xgb.fit(X, y)

permutation = permutation_importance(
    xgb,
    X, y,
    scoring='roc_auc',
    n_repeats=5, n_jobs=-1
)

rank_per = ranking_per(X,permutation)

In [54]:
def plot_permutation_importance(rank_per, num_vars=20):
    # Select top variables
    top = (
        rank_per
        .sort_values(by="importancia_per", ascending=True)
        .tail(num_vars)
    )

    fig = px.bar(
        top,
        x="importancia_per",
        y="variable",
        orientation="h",
        title=f"Top {num_vars} — Permutation Importance",
        labels={"importancia_per": "Importance", "variable": "Variable"},
    )

    fig.update_layout(
        height=40 * num_vars,
        width=900,
        template="plotly_white",
        yaxis=dict(automargin=True)
    )

    fig.show(renderer="iframe")
    return

In [55]:
plot_permutation_importance(rank_per, num_vars=20);

In [56]:
variable_limit_position = 20
# Extract the names of the selected variables
selected_per = rank_per.iloc[0:variable_limit_position].variable

### Ranking summary

In [61]:
def ranking_tot(rank_mi, rank_rfe, rank_per):
    # Merge by 'variable'
    ranking_tot = (rank_mi
        .merge(rank_rfe, on="variable")
        .merge(rank_per, on="variable")
    )
    
    # Sum individual rankings
    ranking_tot["puntos"] = (
        ranking_tot["ranking_mi"] +
        ranking_tot["ranking_rfe"] +
        ranking_tot["ranking_per"]
    )
    
    # Sort by total points (fewer points = better because we sum positions)
    ranking_tot = ranking_tot.sort_values("puntos").reset_index(drop=True)
    
    # Create final ranking 0, 1, 2, ...
    ranking_tot["ranking_tot"] = ranking_tot.index
    
    return ranking_tot


super_ranking = ranking_tot(rank_mi, rank_rfe, rank_per)
super_ranking.head(50)

,variable,importancia_mi,ranking_mi,ranking_rfe,importancia_per,ranking_per,puntos,ranking_tot
0,last_day_all_auto_renew_1.0,0.175821,0,1,0.038723,2,3,0
1,last_day_any_cancel_0.0,0.120650,3,1,0.012176,5,9,1
2,last_day_max_payment_rs,0.062501,12,1,0.026752,3,16,2
3,recency_days_rs,0.055311,17,1,0.059688,1,19,3
4,avg_payment_per_30d_rs,0.076868,10,1,0.004121,9,20,4
5,days_to_expiry_rs,0.043676,20,1,0.087602,0,21,5
6,total_non_cancelled_amount_rs,0.056001,16,1,0.008351,6,23,6
7,registered_via_7.0,0.101461,8,1,0.002402,16,25,7
8,transaction_span_days_rs,0.043306,21,1,0.018521,4,26,8
9,age_unknown,0.118508,4,1,0.000659,21,26,9


In [60]:
import plotly.express as px

# Sort and create index
df_plot = super_ranking.sort_values("puntos", ascending=False).reset_index(drop=True)
df_plot["rank"] = df_plot.index  # 0,1,2,...

fig = px.bar(
    df_plot,
    y="variable",
    x="puntos",
    orientation="h",
    title="<b>Ranking de variables</b>",
    height=800,
    width=700,
    hover_data={"rank": True, "variable": True, "puntos": True},
)

fig.update_layout(
    template="simple_white",
    yaxis=dict(title=""),
    xaxis=dict(title="Puntos"),
)



import plotly.io as pio

print("Renderizador por defecto:", pio.renderers.default)

# Prueba 2: mostrarla dentro de JupyterLab
fig.show(renderer="iframe")


Renderizador por defecto: plotly_mimetype


##  SELECT FINAL METHOD

In [78]:
# Comment / uncomment according to results

# X_preselected = X_mi
# X_preselected = X_rfe
# X_preselected = X_per

X_preselected = X[ranking_tot(rank_mi, rank_rfe, rank_per).variable]

In [79]:
X_preselected.shape

(679672, 79)

## UNSUPERVISED METHODS

### Correlación

In [71]:
# Function to display the result
def strong_correlations(df, lower_lim=0.3, upper_lim=1, drop_dupli=True):
    # Compute the correlation matrix
    c = df.corr().abs()
    # Convert matrix to long format
    c = c.unstack()
    # Reset index and rename columns
    c = pd.DataFrame(c).reset_index()
    c.columns = ['var1', 'var2', 'corr']
    # Convert to dataframe, filter by limits, and sort descending
    c = c.loc[(c['corr'] > lower_lim) & (c['corr'] < upper_lim), :].sort_values(by='corr', ascending=False)
    # Remove duplicated correlations (unless drop_dupli=False)
    c = c if drop_dupli is False else c.drop_duplicates(subset=['corr'])
    # Return output
    return c

#### Compute and review

In [72]:
# Compute
final_corrs = strong_correlations(X_preselected)

# Review aggregated
final_corrs.var1.value_counts()

var1
total_non_cancelled_amount_rs        10
auto_renew_rate_rs                   10
membership_age_days_rs                9
registered_via_9.0                    7
avg_payment_per_30d_rs                7
transaction_span_days_rs              6
has_multi_transaction_day_0.0         6
avg_amount_paid_rs                    6
registered_via_7.0                    6
n_distinct_plan_days_rs               6
last_day_any_auto_renew_0.0           5
avg_plan_days_rs                      5
n_transaction_records_rs              5
total_amount_paid_rs                  5
last_day_max_payment_rs               5
payment_amount_std_rs                 5
recency_days_rs                       5
n_cancellations_rs                    5
n_zero_list_price_transactions_rs     5
n_non_cancelled_transactions_rs       4
n_transaction_days_rs                 4
age_unknown                           4
max_plan_days_rs                      4
gender_Unknown                        4
has_mixed_cancel_day_1.0           

In [73]:
# Review detail
final_corrs.head(50)

,var1,var2,corr
271,total_non_cancelled_amount_rs,total_amount_paid_rs,0.998283
798,n_non_cancelled_transactions_rs,n_transaction_records_rs,0.998136
790,n_non_cancelled_transactions_rs,n_transaction_days_rs,0.993757
712,n_transaction_days_rs,n_transaction_records_rs,0.993491
1009,last_day_any_auto_renew_0.0,last_day_all_auto_renew_0.0,0.991113
20,last_day_all_auto_renew_1.0,last_day_all_auto_renew_0.0,0.987935
1197,has_mixed_cancel_day_0.0,has_mixed_cancel_day_1.0,0.987270
23,last_day_all_auto_renew_1.0,last_day_any_auto_renew_0.0,0.979155
829,avg_plan_days_rs,avg_amount_paid_rs,0.971674
398,age_unknown,gender_Unknown,0.954506


#### Filter variables


## EXPERIMENTS

### A: 108 variables — Baseline

Entrenar un modelo de referencia utilizando todas las variables disponibles, sin aplicar eliminación por correlación ni selección de características.

### B: Eliminación de redundancias extremas

Evaluar un subconjunto de variables eliminando aquellas que presenten correlaciones muy elevadas:

- **Criterio inicial:** correlación absoluta |r| > 0.95.
- Priorizar la conservación de las variables con mayor importancia predictiva.
- Evaluar si la reducción de redundancia mejora el rendimiento o la interpretabilidad del modelo.

### C: Selección supervisada de características

Comparar diferentes subconjuntos según su importancia predictiva:

- Top 30 variables.
- Top 40 variables.
- Top 50 variables.

La selección supervisada debe realizarse **dentro de cada fold de validación cruzada** para evitar *data leakage*.

### Estrategia de evaluación

- Utilizar los mismos folds de validación cruzada para todos los experimentos.
- Comparar los modelos mediante **ROC-AUC** y **Average Precision (AP)**.
- Evaluar tanto el rendimiento predictivo como la complejidad del modelo.
- Reservar el conjunto de test exclusivamente para la evaluación final.

**Objetivo:** identificar el subconjunto más pequeño de características que mantenga un rendimiento predictivo comparable al modelo completo.

## SAVE DATASETS AFTER FEATURE SELECTION

In [82]:
# Define file names
X_preselected_path = project_path / "02_data" / "train" / 'X_selected.pickle'
y_preselected_path = project_path / "02_data" / "train" / 'y_selected.pickle'

# Save files
X_preselected.to_pickle(X_preselected_path)

y_preselected = y.copy()
y_preselected.to_pickle(y_preselected_path)

In [84]:
# Save final feature names
final_feature_names_path = project_path / '05_results/final_feature_names.pickle'

X_preselected.columns.to_series().to_pickle(final_feature_names_path)

In [85]:
len(X_preselected.columns)

79